# Advanced Grouping and Aggregation in Pandas

This notebook walks through customizable aggregation in pandas:
- computing different summary statistics for different columns
- grouping by one or multiple columns
- controlling details like missing groups and unobserved category combinations.

## Setup

In [23]:
import pandas as pd
import numpy as np

## Example DataFrame

We’ll create a small DataFrame that includes:

- continuous (numeric) columns (B, C)

- categorical/grouping columns (E, F, G)

- a column with repeated IDs (A)

- some missing values (to see how dropna=False, count, and size behave)

In [24]:
df = pd.DataFrame({
    "A": ["id1","id1","id2","id3","id3","id4","id5","id5"],
    "B": [10, 12, np.nan, 7, 9, 15, 11, np.nan],
    "C": [1.2, 1.0, 0.9, np.nan, 1.1, 1.4, 1.3, 1.2],
    "E": ["low","low","mid","low","mid","mid", None, "low"],
    "F": ["x","x","y","x","y","y","x","z"],
    "G": ["g1","g2","g1","g1","g2","g2","g1","g2"],
})

df

,A,B,C,E,F,G
0,id1,10.0,1.2,low,x,g1
1,id1,12.0,1.0,low,x,g2
2,id2,NaN,0.9,mid,y,g1
3,id3,7.0,NaN,low,x,g1
4,id3,9.0,1.1,mid,y,g2
5,id4,15.0,1.4,mid,y,g2
6,id5,11.0,1.3,None,x,g1
7,id5,NaN,1.2,low,z,g2


In [39]:
df.mean().apply("B")

TypeError: can only concatenate str (not "int") to str

## Grouping

As discussed in an earlier notebook `.groupby()` splits a dataset into groups based on the unique values of one or more columns. Then calculations can be performed (aggregated) for each group.

Here we'll group by the `E` column and check the mean of the `B` column for each unique value of `E`.

In [3]:
df.groupby("E")["B"].mean()

E
low     9.666667
mid    12.000000
Name: B, dtype: float64

By default, Pandas does not include `NaN` as a unique value.
If you want missing categories included as their own group, use:

In [4]:
df.groupby("E", dropna=False)["B"].mean()

E
low     9.666667
mid    12.000000
NaN    11.000000
Name: B, dtype: float64

## Size vs Count

When summarizing group sizes:

- `.size()` = number of rows in the group (includes missing values in other columns)

- `.count()` = number of non-missing values in the chosen column

In [5]:
df

,A,B,C,E,F,G
0,id1,10.0,1.2,low,x,g1
1,id1,12.0,1.0,low,x,g2
2,id2,NaN,0.9,mid,y,g1
3,id3,7.0,NaN,low,x,g1
4,id3,9.0,1.1,mid,y,g2
5,id4,15.0,1.4,mid,y,g2
6,id5,11.0,1.3,None,x,g1
7,id5,NaN,1.2,low,z,g2


In [5]:
print(df.groupby("E", dropna=False)["B"].size())
print(df.groupby("E", dropna=False)["B"].count())

E
low    4
mid    3
NaN    1
Name: B, dtype: int64
E
low    3
mid    2
NaN    1
Name: B, dtype: int64


## Aggregate

We can use the `.aggregate()` or `.agg()` function to directly calculate multiple summary statistics across groups.

- `.mean()`: Calculate the average of values.
- `.sum()`: Calculate the sum of values.
- `.min()`: Find the minimum value.
- `.max()`: Find the maximum value.
- `.std()`: Calculate the standard deviation.
- `.median()`: Calculate the median value.
- `.count()`: Count the number of non-missing values.
- `.nunique()`: Count the number of unique values.

For example, we could group the data by the `E` column and then ask for the mean of the `B` column for each group with `.agg()`. The syntax is:
```
df.groupby(<column to group by>).agg({<column>: <summary function>})
```

In [6]:
df.groupby("E", dropna=False).agg(
    {'B': 'mean'}
      )

,B
E,
low,9.666667
mid,12.000000
NaN,11.000000


This could be more simply accomplished with the syntax we covered earlier:

In [ ]:
df.groupby("E", dropna=False)['B'].mean()

But the real power of `.agg()` comes from flexibility in applying multiple functions. Here we'll apply a list of aggregation functions ([full list of possibilities here in Pandas documentation](https://pandas.pydata.org/docs/user_guide/groupby.html#built-in-aggregation-methods)) to the B and C columns for each group within the E column.

In [6]:
df.groupby("E", dropna=False)[['B', 'C']].agg(['mean', 'std'])

B                   C          
          mean       std      mean       std
E                                           
low   9.666667  2.516611  1.133333  0.115470
mid  12.000000  4.242641  1.133333  0.251661
NaN  11.000000       NaN  1.300000       NaN

To apply specific functions to specific columns, you can do **named aggregation**, where you provide a name for each resulting column with a tuple of the `(<column name>, <aggregation function>)`. As you can see, it can get quite complex!

In [7]:
df_E_summary_info = (
    df.groupby("E", dropna=False)
      .aggregate(
          B_avg=('B', 'mean'),
          B_std=('B', 'std'),
          B_sem=('B', 'sem'),
          C_avg=('C', 'mean'),
          C_sem=('C', 'sem'),
          B_numrows=('B', 'size'),
          B_nonmissing=('B', 'count'),
          F_nunique=('F', 'nunique'),
          G_nunique=('G', 'nunique'),
          A_nunique=('A', 'nunique')
      )
      .reset_index()
)

df_E_summary_info # we do the above statistics for all the values in the column 'E'('low', 'mid', 'NaN')

,E,B_avg,B_std,B_sem,C_avg,C_sem,B_numrows,B_nonmissing,F_nunique,G_nunique,A_nunique
0,low,9.666667,2.516611,1.452966,1.133333,0.066667,4,3,2,2,3
1,mid,12.000000,4.242641,3.000000,1.133333,0.145297,3,2,1,2,3
2,NaN,11.000000,NaN,NaN,1.300000,NaN,1,1,1,1,1


### Grouping by Multiple Columns

You can group by more than one column to get “subgroup” summaries. The following dataframe creates groups of rows for combinations of unique values within the E and F columns. This returns what's called a "multi-index" Series with 2 levels of indices.

In [8]:
df.groupby(["E", "F"], dropna=False)["B"].mean()

E    F
low  x     9.666667
     z          NaN
mid  y    12.000000
NaN  x    11.000000
Name: B, dtype: float64

We won't be dealing with multi-index data structures in this course, so you can convert things to a regular DataFrame with `.reset_index()`.

In [9]:
df.groupby(["E", "F"], dropna=False)["B"].mean().reset_index()

,E,F,B
0,low,x,9.666667
1,low,z,NaN
2,mid,y,12.000000
3,NaN,x,11.000000


### Practice Questions

Question 1) Create a summary table grouped by F that includes:

- B_avg, B_std, B_nonmissing, B_numrows

Question 2) Group by E and G, and compute:

- C_avg, C_sem, and A_nunique

In [15]:
df

,A,B,C,E,F,G
0,id1,10.0,1.2,low,x,g1
1,id1,12.0,1.0,low,x,g2
2,id2,NaN,0.9,mid,y,g1
3,id3,7.0,NaN,low,x,g1
4,id3,9.0,1.1,mid,y,g2
5,id4,15.0,1.4,mid,y,g2
6,id5,11.0,1.3,None,x,g1
7,id5,NaN,1.2,low,z,g2


In [10]:
question_1 = (
    df.groupby("F", dropna=False).aggregate(
    B_avg=("B", "mean"), 
    B_std=("B", "std"), 
    B_nonmissing=("B", "count"), 
    B_numrows=("B", "size")
    )
    .reset_index()
)
question_1

,F,B_avg,B_std,B_nonmissing,B_numrows
0,x,10.0,2.160247,4,4
1,y,12.0,4.242641,2,3
2,z,NaN,NaN,0,1


In [21]:
question_2 = (df.groupby(["E","G"])
              .agg(
                  C_avg=("C", "mean"),
                  C_sem=("C", "sem"),
                  A_nunique=("A", "nunique")
              )
              .reset_index()
)
question_2

,E,G,C_avg,C_sem,A_nunique
0,low,g1,1.20,NaN,2
1,low,g2,1.10,0.10,2
2,mid,g1,0.90,NaN,1
3,mid,g2,1.25,0.15,2
